### Interactive notebook to add/modify things in local database for testing purposes

In [7]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from models import Base, Recommendation, Location, User, UserProfile
from enums import RiskLevel, Severity, Trigger
from database import engine, Session
from ast import literal_eval

In [2]:
def clear_db():
    """clear out all tables currently in the database"""
    Base.metadata.drop_all(engine)

def add_schema():
    """add the tables defined in models.py to the database"""
    Base.metadata.create_all(engine)

def fill_recs():
    """Fill the recs table with the data from rec_defs.csv.
    The recs table is first empied so that the table in the database matches the csv file exactly.
    """
    recs_df = pd.read_csv("rec_defs.csv", converters={"drivers": literal_eval})
    recs_df["sources"] = recs_df["sources"].apply(lambda x: x.split("|"))
    recs_df.insert(0, "rec_id", recs_df.index)

    recs_df.to_sql("recs", con=engine, if_exists="append", index=False)

def insert_mock_user_1():
    """Insert a mock 'first' user + profile into the database.
    Also interts mock location data for the user.
    """
    with Session() as db:
        first_loc = Location(zip_code=34201, latitude=27.4, longitude=-82.5)
        db.add(first_loc)
        db.flush()

        first_user = User(username="adam")
        db.add(first_user)
        db.flush()

        first_profile = UserProfile(
            user_id=first_user.user_id,
            first_name="Adam",
            last_name="Prime",
            zip_code=first_loc.zip_code,
            severity=Severity.MILD_PERSISTENT,
            triggers=[Trigger.POLLEN_OUTDOOR_MOLD, Trigger.COLD_AIR, Trigger.EXERCISE]
        )
        db.add(first_profile)

        db.commit()
    

